# Projection-based WF-in-DFT embedding

A clean end-to-end run of $\mu$-shift projection embedding: a cheap DFT calculation on the whole molecule, then a correlated wavefunction on a fragment of it, with the environment kept at the DFT level.

The physics follows Manby *et al.*,
[JCTC **8**, 2564 (2012)](https://doi.org/10.1021/ct300544e), and the energy expression is
Eq. 8 of Goodpaster *et al.*,
[JCP **140**, 18A507 (2014)](https://doi.org/10.1063/1.4864040).

**Roadmap**

1. Partition the global DFT energy into fragment, environment and cross terms.
2. Choose which occupied orbitals belong to the fragment (`code/active_space.py`).
3. Build the embedding potential $v_\text{emb}$ and the environment projector $\hat P_B$.
4. Validate the machinery with DFT-in-DFT, which must reproduce the global DFT energy exactly.
5. Swap the fragment solver for HF, CISD, CCSD and CASCI.
6. Check the exact limit: with the whole molecule as the fragment, every WF-in-DFT number
   must collapse onto the corresponding whole-molecule wavefunction energy.

**Three things that silently give wrong answers**, all of which bit notebook 23 and each of
which is called out at the point it matters below:

- the projector must span the *occupied* environment orbitals only;
- the embedding correction contracts $v_\text{emb}$ with the *DFT* fragment density, not with
  the solver's density;
- an active space must be chosen from the *post-embedding* orbitals, never by reusing
  orbital indices from the global calculation.

Section 6 also looks at what deleting the level-shifted orbitals does and does not buy, which
turns out to be a cost and resource question rather than a correctness one.

In [1]:
import numpy as np
from pyscf import gto, scf, lo, ci, cc, mcscf

## 1. Splitting the DFT energy in two

Run KS-DFT on the whole molecule and split its occupied orbitals into two disjoint sets, a
fragment $A$ and an environment $B$. Because the sets are disjoint and the orbitals are
orthonormal, the total density splits exactly:

$$\gamma = \gamma_A + \gamma_B .$$

The energy does not split exactly, because it is not linear in the density. Define the
leftover as the non-additive energy:

$$E_\text{nad}[\gamma_A,\gamma_B] \;\equiv\; E_\text{DFT}[\gamma_A+\gamma_B] - E_\text{DFT}[\gamma_A] - E_\text{DFT}[\gamma_B],$$

so that

$$E_\text{DFT}[\gamma] = E_\text{DFT}[\gamma_A] + E_\text{DFT}[\gamma_B] + E_\text{nad}[\gamma_A,\gamma_B].$$

Nothing is approximated yet; $E_\text{nad}$ is defined by this equation. It collects the
inter-subsystem Coulomb repulsion and the non-additive exchange-correlation energy.

The reason to build the partition from *orbitals* rather than from densities alone is that
there is **no non-additive kinetic energy term** here. Both subsystems are described by
orthonormal orbital sets drawn from one Slater determinant, so the kinetic energy is additive
by construction. Approximating the non-additive kinetic energy is the main error source in
pure density-based (frozen-density) embedding, and projection-based embedding avoids it
entirely. That is what makes DFT-in-DFT exact here, which we will verify numerically.

In the code below, energies are electronic only (`energy_elec`, which excludes nuclear
repulsion), and each subsystem density is evaluated in the field of *all* the nuclei, so
$E_\text{nuc}$ enters the total exactly once:

- `E_act_dft` $= E_\text{DFT}[\gamma_A]$
- `E_env` $= E_\text{DFT}[\gamma_B]$
- `E_cross` $= E_\text{nad}[\gamma_A,\gamma_B]$

### The system

Methanol, with the hydroxyl group as the fragment. `6-31G` and `b3lyp` keep every step in this
notebook to a second or two, so the whole thing can be re-run while changing the partition.

In [2]:
geometry = [
    ("O", (-0.6582, -0.0067,  0.1730)),
    ("H", (-1.1326, -0.0311, -0.6482)),
    ("C", ( 0.7031,  0.0083, -0.1305)),
    ("H", ( 0.9877,  0.8943, -0.7114)),
    ("H", ( 1.0155, -0.8918, -0.6742)),
    ("H", ( 1.2001,  0.0363,  0.8431)),
]

basis_set = "6-31G"
xc        = "b3lyp"
charge    = 0
spin      = 0          # 2S
max_memory = 10_000    # Mb

# the fragment: the hydroxyl O and H (0-based atom indices)
active_atm_idx = [0, 1]
n_occ_active   = 3     # occupied orbitals given to the fragment -> its electron count
n_vir_active   = 3     # extra virtuals in the fragment block -> sets the CAS size
mu_val         = 1e6   # level shift

mol = gto.Mole(
    atom=geometry,
    basis=basis_set,
    charge=charge,
    spin=spin,
    max_memory=max_memory,
).build()

mol.nao, mol.nelec

(26, (9, 9))

In [3]:
## the cheap calculation on the whole molecule: everything downstream is built from this
global_scf = scf.RKS(mol, xc=xc)
global_scf.kernel()

## whole-molecule wavefunction references, for context later on
mf_hf = scf.RHF(mol).run()
ci_full = ci.CISD(mf_hf).run()
cc_full = cc.CCSD(mf_hf).run()

print(f"\nwhole molecule: B3LYP {global_scf.e_tot:.8f}   RHF {mf_hf.e_tot:.8f}   "
      f"CISD {ci_full.e_tot:.8f}   CCSD {cc_full.e_tot:.8f}")

converged SCF energy = -115.675712801882
converged SCF energy = -114.985033959313
E(RCISD) = -115.2044777799447  E_corr = -0.2194438206315685
E(CCSD) = -115.2206710296384  E_corr = -0.2356370703252617

whole molecule: B3LYP -115.67571280   RHF -114.98503396   CISD -115.20447778   CCSD -115.22067103


## 2. Which orbitals belong to the fragment

`active_space.py` scores every MO by how much of it sits on the target atoms, using a
**Löwdin population**. Orthogonalise the AO basis with $S^{1/2}$, so that squared coefficients
genuinely partition an orbital, and sum them over the target atoms' AOs:

$$w_i \;=\; \sum_{\mu \in A} \left[ (S^{1/2} C)_{\mu i} \right]^2 \;\in\; [0,1].$$

Because the basis is orthogonalised first, $\sum_\mu$ over all atoms gives exactly 1 for every
orbital, so $w_i$ reads directly as "the fraction of orbital $i$ living on the fragment". A raw
Mulliken sum over the same AOs is not bounded this way and can even go negative. Core $1s$ AOs
are dropped for elements beyond helium so that tight cores cannot dominate the score; H keeps
its $1s$, which is its valence shell.

Occupied and virtual orbitals are ranked separately, and no MO-index window is used: a strongly
fragment-localised orbital high in the virtual space is usually a compact $\sigma^*$, exactly
what bond breaking needs, and `orbital_spread` (the RMS extent $\sqrt{\langle r^2\rangle - \langle r\rangle^2}$)
is the honest way to reject genuinely diffuse Rydberg-like orbitals instead.

**Only the occupied partition affects the embedding.** The fragment occupied orbitals fix the
subsystem electron count, and the environment occupied orbitals build the projector. The
selected *virtuals* never enter the embedded SCF, because after embedding the subsystem has its
own virtual space; they only fix how big a CAS we hand to a post-embedding solver.

### Why localise first

Canonical KS orbitals are delocalised, so no single one of them is "the O–H bond": the split is
fuzzy and the fragment ends up sharing orbitals with the environment. A unitary rotation
*within* the occupied block leaves $\gamma$, and therefore the global DFT energy, completely
unchanged, but it makes the orbitals atom-centred and the partition clean. That is free
accuracy, so do it. Below, Pipek-Mezey is applied separately to the occupied and virtual
blocks, which keeps `mo_occ` meaningful column by column.

In [5]:
## localise inside the occupied and virtual blocks separately, so that mo_occ still
## describes column i, and the total density is untouched
occ_mask = global_scf.mo_occ > 0
C_loc = global_scf.mo_coeff.copy()
C_loc[:,  occ_mask] = lo.PipekMezey(mol, global_scf.mo_coeff[:,  occ_mask]).kernel()
C_loc[:, ~occ_mask] = lo.PipekMezey(mol, global_scf.mo_coeff[:, ~occ_mask]).kernel()

## a unitary rotation within the occupied block must leave the density alone
assert np.allclose(
    global_scf.make_rdm1(mo_coeff=C_loc, mo_occ=global_scf.mo_occ),
    global_scf.make_rdm1(),
    atol=1e-9,
), "localisation changed the density"


### What the partition object holds

`select_active_space` returns the orbitals reordered as

$$C_\text{active} = [\;\underbrace{\text{env occ}}_{n_\text{core}}\;|\;\underbrace{\text{frag occ}\;|\;\text{frag vir}}_{n_\text{cas}}\;|\;\text{remaining vir}\;]$$

so that `act_cols` and `env_cols` index *this* ordering. Two attributes matter most:

- `env_occ_cols` — the occupied environment orbitals, which build the projector;
- `n_env_mo` — how many orbitals get level-shifted, and so how many a correlated solver
  must discard later.

`orig_active_idxs` records the fragment orbitals in the original numbering. It is for
diagnostics only: those indices refer to the *global* orbital set and are meaningless as a
selector once the embedded SCF has produced its own orbitals.

`space.densities()` builds $\gamma$, $\gamma_A$ and $\gamma_B$ from the same reordered
coefficients, and asserts that they are additive and hold the right electron counts.

In [7]:
from nbed.embedding import get_mu_projector, get_embedding_potential
from nbed.active_space import lowdin_populations, orbital_spread

In [8]:
C = C_loc
# C =  global_scf.mo_coeff.copy()


per_atom, population = lowdin_populations(mol, C, active_atm_idx, drop_core_1s=True)
spread = orbital_spread(mol, C)

eligible = np.ones_like(population, dtype=bool)

# max_spread: Reject orbitals more diffuse than this, in BOHR.
max_spread = 2
if max_spread is not None:
    eligible &= spread <= max_spread


occ_pool = np.where((global_scf.mo_occ > 0) & eligible)[0]
vir_pool = np.where((global_scf.mo_occ == 0) & eligible)[0]

if len(occ_pool) < n_occ_active or len(vir_pool) < n_vir_active:
    raise ValueError(
        f"asked for {n_occ_active} occupied and {n_vir_active} virtual "
        f"orbitals but only {len(occ_pool)} and {len(vir_pool)} are "
        "eligible; relax max_spread or shrink the fragment"
    )

## get most important occupied and virtual orbitals
pick_occ = np.sort(occ_pool[np.argsort(-population[occ_pool])[:n_occ_active]])
pick_vir = np.sort(vir_pool[np.argsort(-population[vir_pool])[:n_vir_active]])

In [9]:
core_idx = np.setdiff1d(np.where(global_scf.mo_occ > 0)[0], pick_occ)
rest_vir = np.setdiff1d(np.where(global_scf.mo_occ == 0)[0], pick_vir)
re_idx = np.concatenate([core_idx, pick_occ, pick_vir, rest_vir])


ncore = len(core_idx)
ncas = n_occ_active + n_vir_active
act_cols = np.arange(ncore, ncore + ncas)
env_cols = np.setdiff1d(np.arange(len(re_idx)), act_cols)

## new order is [core_fixed, active, fixed_virtual]
mo_occ_full_reidx = global_scf.mo_occ[re_idx]
C_full_reidx = C[:, re_idx]
mo_occ_act = mo_occ_full_reidx.copy()
mo_occ_act[env_cols] = 0
mo_occ_env = mo_occ_full_reidx.copy()
mo_occ_env[act_cols] = 0

nelec_act = (int((mo_occ_act > 0).sum()), int((mo_occ_full_reidx[act_cols] > 1).sum()))
nelec_env = (int((mo_occ_env > 0).sum()), int((mo_occ_full_reidx[env_cols] > 1).sum()))

nelec_act, nelec_env, mol.nelec


((3, 3), (6, 6), (9, 9))

In [10]:
dm_full = global_scf.make_rdm1()
dm_act = global_scf.make_rdm1(mo_coeff=C_full_reidx, mo_occ=mo_occ_act)
dm_env = global_scf.make_rdm1(mo_coeff=C_full_reidx, mo_occ=mo_occ_env)

np.allclose(dm_full, dm_act + dm_env)

True

In [11]:
coords   = mol.atom_coords(unit="A")
atm_list = [mol.atom_pure_symbol(i) for i in range(mol.natm)]

mol_act = gto.Mole(
    atom=zip(atm_list, coords),
    unit="A",
    basis=mol.basis,
    charge=mol.charge + mol.nelectron - sum(nelec_act),
    spin=nelec_act[0] - nelec_act[1],
    max_memory=max_memory,
).build()



assert mol_act.nelectron == sum(nelec_act)

## PITFALL 1: occupied environment orbitals only
C_env_occ = C_full_reidx[:, mo_occ_env>0]
P_env_ao  = get_mu_projector(C_env_occ, mol_act.intor("int1e_ovlp"))

G_EMB     = get_embedding_potential(global_scf, dm_full, dm_act)
v_emb = mu_val*P_env_ao + G_EMB

## the projector is idempotent in the S metric, and blind to the fragment density
assert np.isclose(np.einsum("ij,ji->", dm_act, P_env_ao), 0.0, atol=1e-9), \
    "fragment density leaks into the environment orbitals"
print(f"tr[dm_env . P_env] = {np.einsum('ij,ji->', dm_env, P_env_ao):.6f}"
      f"   (= number of environment electrons: {sum(mo_occ_env)})")

tr[dm_env . P_env] = 12.000000   (= number of environment electrons: 12.0)


In [12]:
E_nuclear   = global_scf.energy_nuc()
E_DFT_cheap = global_scf.energy_elec(dm=dm_full)[0]
E_env_cheap = global_scf.energy_elec(dm=dm_env)[0]
E_act_cheap = global_scf.energy_elec(dm=dm_act)[0]
E_cross     = E_DFT_cheap - E_env_cheap - E_act_cheap

## PITFALL 2: v_emb against the DFT fragment density, never the solver's density
emb_corr = np.einsum("ij,ji->", dm_act, v_emb)

## everything that does not depend on which solver we use for the fragment
classical = E_env_cheap + E_cross - emb_corr

## the partition of the DFT energy is exact, so this must reproduce the global result
E_full = global_scf.energy_tot(dm=dm_full)
assert np.isclose(E_act_cheap + E_env_cheap + E_cross + E_nuclear, E_full)

print(f"E_act (DFT)  {E_act_cheap:14.8f}")
print(f"E_env (DFT)  {E_env_cheap:14.8f}")
print(f"E_cross      {E_cross:14.8f}")
print(f"E_nuclear    {E_nuclear:14.8f}")
print(f"emb_corr     {emb_corr:14.8f}   (mu part: "
      f"{mu_val*np.einsum('ij,ji->', dm_act, P_env_ao):.2e})")
print(f"sum          {E_full:14.8f}  == global B3LYP {global_scf.e_tot:.8f}")

E_act (DFT)    -47.09093962
E_env (DFT)   -141.89549269
E_cross         32.52019260
E_nuclear       40.79052691
emb_corr        32.64953864   (mu part: 1.55e-11)
sum           -115.67571280  == global B3LYP -115.67571280


In [38]:
### DFT-in-DFT

XC = global_scf.xc # same as before
# XC =  "wb97x"    # better functional!

ks_emb = scf.RKS(mol_act, xc=XC)

hcore_std = ks_emb.get_hcore()
hcore_mod = hcore_std + v_emb
ks_emb.get_hcore = lambda *args: hcore_mod

ks_emb.kernel()
dm_emb_ks = ks_emb.make_rdm1()

E_dft_in_dft = ks_emb.e_tot + classical

print(f"\nDFT-in-DFT      {E_dft_in_dft:.10f}")
print(f"global B3LYP    {global_scf.e_tot:.10f}")
print(f"difference      {E_dft_in_dft - global_scf.e_tot:+.2e}")
print(f"\nmu leakage into the embedded density: "
      f"{mu_val*np.einsum('ij,ji->', dm_emb_ks, P_env_ao):.2e}")

converged SCF energy = 26.3491257794858

DFT-in-DFT      -115.6757129493
global B3LYP    -115.6757128019
difference      -1.47e-07

mu leakage into the embedded density: 1.47e-07


In [39]:
## the HF reference for the correlated solvers, carrying the same h_emb
n_env_mo = len(env_cols)

rhf_act = scf.RHF(mol_act)
rhf_act.get_hcore = lambda *args: hcore_mod
rhf_act.kernel()

E_hf_in_dft = rhf_act.e_tot + classical

converged SCF energy = 26.6313553616853


In [40]:
## how much does deleting them actually change? run CISD both ways.
ci_keep_all = ci.CISD(rhf_act).run()   # all nao orbitals, shifted ones included
leak = np.einsum("ij,ji->", ci_keep_all.make_rdm1(ao_repr=True), mu_val*P_env_ao)
print(f"\nmu val:    {mu_val:.0e}")
print(f"CISD energy: {ci_keep_all.e_tot + classical:.9f}")
print(f"leakage tr[g_CISD.muP]  {leak:.2e}")

E(RCISD) = 26.54908082578597  E_corr = -0.0822745358992929

mu val:    1e+06
CISD energy: -115.475757903
leakage tr[g_CISD.muP]  2.11e-07


In [48]:
## correlate every kept orbital of the fragment
ci_obj = ci.CISD(rhf_act).run()
cc_obj = cc.CCSD(rhf_act).run()

E_cisd_in_dft = ci_obj.e_tot + classical
E_ccsd_in_dft = cc_obj.e_tot + classical

## PITFALL 3: the CAS comes from the embedded orbitals, not from global orbital indices
ncas    = len(act_cols) 
nelecas = nelec_act
mycas = mcscf.CASCI(rhf_act, ncas, nelecas)
assert mycas.ncore == 0, "ncore > 0: the frozen core also carries v_emb, handle energy_core"
mycas.kernel()

E_casci_in_dft = mycas.e_tot + classical

print(f"\nCISD  e_corr {ci_obj.e_corr:.6f}")
print(f"CCSD  e_corr {cc_obj.e_corr:.6f}")
print(f"CASCI({ncas}o,{sum(nelecas)}e) ncore={mycas.ncore}")

E(RCISD) = 26.54908082578597  E_corr = -0.0822745358992929
E(CCSD) = 26.54736096710396  E_corr = -0.08399439458130317
CASCI E = 26.6285274569992  E(CI) = -14.1619994499505  S^2 = 0.0000000

CISD  e_corr -0.082275
CCSD  e_corr -0.083994
CASCI(6o,6e) ncore=0


In [49]:
frag = ", ".join(f"{mol.atom_symbol(a)}{a}" for a in active_atm_idx)
print(f"fragment: {frag}   CAS({sum(mycas.nelecas)}e, {mycas.ncas}o)   "
      f"basis {basis_set}   environment functional {xc}\n")

print(f"{'whole molecule':<28}{'':>16}")
for name, e in [("B3LYP", global_scf.e_tot), ("RHF", mf_hf.e_tot),
                ("CISD", ci_full.e_tot), ("CCSD", cc_full.e_tot)]:
    print(f"  {name:<24}{e:16.8f}")

print(f"\n{'embedded':<28}{'':>16}")
for name, e in [("DFT-in-DFT", E_dft_in_dft), ("HF-in-DFT", E_hf_in_dft),
                (f"CASCI({mycas.ncas},{sum(mycas.nelecas)})-in-DFT", E_casci_in_dft),
                ("CISD-in-DFT", E_cisd_in_dft), ("CCSD-in-DFT", E_ccsd_in_dft)]:
    print(f"  {name:<24}{e:16.8f}")

fragment: O0, H1   CAS(6e, 6o)   basis 6-31G   environment functional b3lyp

whole molecule                              
  B3LYP                      -115.67571280
  RHF                        -114.98503396
  CISD                       -115.20447778
  CCSD                       -115.22067103

embedded                                    
  DFT-in-DFT                 -115.67571295
  HF-in-DFT                  -115.39348337
  CASCI(6,6)-in-DFT          -115.39631127
  CISD-in-DFT                -115.47575790
  CCSD-in-DFT                -115.47747776


In [81]:
geometry = [
    ("Fe", (0.0, 0.0, 0.0)),
    ("H"  , (0.0, 0.0, 1.0)),
    ("H"  , (0.0, 0.0, -1.0)),
]

basis_set = "cc-pvdz"
charge    = 1
spin      = 3         # 2S
max_memory = 10_000    # Mb

mol = gto.Mole(
    atom=geometry,
    basis=basis_set,
    charge=charge,
    spin=spin,
    max_memory=max_memory,
).build()

mol.nao, mol.nelec

(53, (15, 12))

In [82]:
uhf_obj = scf.UHF(mol).run()


WARN: HOMO 0.131994272624346 >= LUMO 0.0808736126588149


WARN: HOMO -2.17392959664878 >= LUMO -3.95583935858196

converged SCF energy = -1262.5872783938  <S^2> = 3.7634729  2S+1 = 4.0067308


In [86]:
# dm_a_ao, dm_b_ao = uhf_obj.make_rdm1()

# Ca = uhf_obj.mo_coeff[0]
# Cb = uhf_obj.mo_coeff[1]
# Sao = uhf_obj.get_ovlp()

# dm_a_mo = Ca.T @ Sao @ dm_a_ao @ Sao @ Ca
# dm_b_mo = Cb.T @ Sao @ dm_b_ao @ Sao @ Cb
# print(np.around(dm_a_mo, 15))
# print(np.around(dm_b_mo, 15))